In [3]:
import json

with open("/lustre/groups/ml01/workspace/ot_perturbation/models/otfm/pbmc_new_donor/all_preds.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [13]:
data_64 = [el for el in data if "65_pred" in el]

In [14]:
len(data_64)

30

In [16]:
with open("/lustre/groups/ml01/workspace/ot_perturbation/models/otfm/pbmc_new_donor/pred_files_64.json", "w") as f:
    json.dump(data_64, f)

In [1]:
import json

with open("/lustre/groups/ml01/workspace/ot_perturbation/models/otfm/pbmc_new_donor/all_preds.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [39]:
data_64 = [el for el in data if "65_pred" in el]

In [40]:
len(data_64)

360

In [41]:
import pandas as pd
df = pd.DataFrame(data_64, columns=["name"])
df.head()

,name
0,charmed-dragon-45_Donor1_IFN-gamma_65_preds.h5ad
1,fallen-plasma-117_Donor4_FasL_65_preds.h5ad
2,absurd-puddle-164_Donor6_ADSF_65_preds.h5ad
3,swept-eon-260_Donor10_CD27L_65_preds.h5ad
4,winter-terrain-310_Donor12_IFN-gamma_65_preds....


In [42]:
df["donor"] = df.apply(lambda x: x["name"].split("_")[1], axis=1)
df["cytokine"] = df.apply(lambda x: x["name"].split("_")[2], axis=1)

In [43]:
df["donor"].value_counts()

donor
Donor1     30
Donor4     30
Donor6     30
Donor10    30
Donor12    30
Donor11    30
Donor9     30
Donor8     30
Donor3     30
Donor7     30
Donor5     30
Donor2     30
Name: count, dtype: int64

In [44]:
df["cytokine"].value_counts()

cytokine
IFN-gamma     36
FasL          36
ADSF          36
CD27L         36
IFN-omega     36
OX40L         36
BAFF          36
M-CSF         36
IL-1Ra        36
IL-32-beta    36
Name: count, dtype: int64

In [35]:
df["condition"] = df.apply(lambda x: x["donor"] + "_" + x["cytokine"], axis=1)

In [36]:
df["condition"].value_counts()

condition
Donor10_ADSF          4
Donor2_ADSF           4
Donor2_IL-1Ra         4
Donor10_IL-32-beta    4
Donor2_BAFF           4
                     ..
Donor9_IFN-gamma      3
Donor4_ADSF           3
Donor11_IL-32-beta    3
Donor12_IFN-omega     3
Donor5_IFN-omega      3
Name: count, Length: 120, dtype: int64

In [37]:
df[df["condition"]=="Donor5_IFN-omega"]

,name,donor,cytokine,condition
306,restful-darkness-141_Donor5_IFN-omega_65_preds...,Donor5,IFN-omega,Donor5_IFN-omega
341,fiery-water-140_Donor5_IFN-omega_65_preds.h5ad,Donor5,IFN-omega,Donor5_IFN-omega
355,neat-glitter-139_Donor5_IFN-omega_65_preds.h5ad,Donor5,IFN-omega,Donor5_IFN-omega


In [38]:
df[df["condition"]=="Donor10_ADSF"]

,name,donor,cytokine,condition
55,grateful-music-265_Donor10_ADSF_81_preds.h5ad,Donor10,ADSF,Donor10_ADSF
212,swept-eon-260_Donor10_ADSF_65_preds.h5ad,Donor10,ADSF,Donor10_ADSF
300,denim-jazz-261_Donor10_ADSF_65_preds.h5ad,Donor10,ADSF,Donor10_ADSF
347,firm-dew-259_Donor10_ADSF_65_preds.h5ad,Donor10,ADSF,Donor10_ADSF


In [1]:
import json

with open("/lustre/groups/ml01/workspace/ot_perturbation/models/otfm/pbmc_new_donor/all_preds.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [3]:
data_64 = [el for el in data if "64_" in el]

In [4]:
len(data_64)

30

In [8]:
len([el for el in data if "65_" in el]) # we might have to rerun if it turns out to be 373 in the mean model!

373

In [1]:
import scanpy as sc

/home/icb/dominik.klein/mambaforge/envs/cfp/lib/python3.11/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module


In [3]:
import os
pred_file = "absurd-puddle-164_Donor6_ADSF_65_preds.h5ad"
complete_pred_file = os.path.join("/lustre/groups/ml01/workspace/ot_perturbation/models/otfm/pbmc_new_donor", pred_file)


In [4]:
adata_pred = sc.read_h5ad(complete_pred_file)
adata_pred.X = adata_pred.layers["X_recon"]
cytokine = pred_file.split("_")[-3]
donor = pred_file.split("_")[-4]
out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/metrics_new_donor_different_k"
adata_full = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/pbmc_with_pca.h5ad")
adata_ood_true = adata_full[(adata_full.obs["donor"] == donor) & (adata_full.obs["cytokine"]==cytokine)]
adata_ctrl = adata_full[(adata_full.obs["cytokine"]=="PBS") & (adata_full.obs["donor"]==donor)]



In [5]:
import pickle
with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/degs_different_top_k.pkl", "rb") as pickle_file:
    deg_genes = pickle.load(pickle_file)



In [13]:
import functools
import os
import sys
import traceback
from typing import Dict, Literal, Optional, Tuple
import cfp
import scanpy as sc
import numpy as np
import functools
from ott.solvers import utils as solver_utils
import optax
from omegaconf import OmegaConf
from typing import NamedTuple, Any
import hydra
import wandb
import anndata as ad
import pandas as pd
import os
import pickle
from cfp.preprocessing import transfer_labels, compute_wknn
from cfp.training import ComputationCallback
from numpy.typing import ArrayLike
from cfp.metrics import compute_r_squared, compute_e_distance, compute_scalar_mmd, compute_sinkhorn_div
from cfp.metrics import compute_r_squared, compute_e_distance, compute_metrics_fast
from cfp.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca




def compute_metrics(adata_ref: ad.AnnData, adata_pred: ad.AnnData, deg_dict: dict, adata_ood_true: ad.AnnData, adata_ctrl: ad.AnnData, n_neighbors: int=1, cell_type_col: str = "cell_type_new", min_cells_for_dist_metrics: int = 50) -> dict:
    dict_to_log = {}
    compute_wknn(ref_adata=adata_ref, query_adata=adata_pred, n_neighbors=n_neighbors, ref_rep_key="X_pca", query_rep_key="X_pca_for_ct_transfer")
    transfer_labels(query_adata=adata_pred, ref_adata=adata_ref, label_key=cell_type_col)

    deg_r_sq = {}
    for k in deg_genes.keys():
        donor_deg_dict = {k: v for k, v in deg_genes[k].items() if (k.startswith(donor) and k.endswith(f"_{cytokine}"))}
        deg_r_sq[k] = {}
        for ct_cyto in donor_deg_dict.keys():
            cell_type = ct_cyto.split("_")[1]
            adata_true_ct = adata_ood_true[(adata_ood_true.obs[f"{cell_type_col}"]==cell_type)]
            adata_pred_ct = adata_pred[adata_pred.obs[f"{cell_type_col}_transfer"]==cell_type]
            if adata_pred_ct.n_obs == 0:
                continue
        
            deg_mask = [True if el in donor_deg_dict[ct_cyto] else False for el in adata_ood_true.var_names]
            deg_true_decoded = adata_true_ct[:,deg_mask].X.toarray()
            deg_pred_decoded = adata_pred_ct[:,deg_mask].X
            deg_r_sq[k][f"deg_decoded_r_squared_{cell_type}"] = compute_r_squared(deg_true_decoded, deg_pred_decoded)
        
      
    return deg_r_sq


In [10]:
adata_full = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/pbmc_with_pca.h5ad")
with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/idcs_to_keep.pkl", "rb") as pickle_file:
    idcs_to_keep = pickle.load(pickle_file)

adata_ref = adata_full[adata_full.obs_names.isin(idcs_to_keep)]
project_pca(query_adata=adata_pred, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")

for k in deg_genes.keys():
    adata_pred.obs["cytokine"] = cytokine
    adata_pred.obs["donor"] = donor
    adata_pred.var_names=adata_ctrl.var_names 
    cond = f'{donor}_{cytokine}'
    out = compute_metrics(adata_ref=adata_ref, adata_pred=adata_pred, deg_dict=deg_genes, adata_ood_true=adata_ood_true, adata_ctrl=adata_ctrl)
    df = pd.DataFrame.from_dict(out, orient="index")
    df["cond"] = cond
    df["wandb_name"] = pred_file.split("_")[0]
    df["cytokine_in_train"] = adata_pred.uns["cytokine_in_train"]
    df["cytokine_in_train"] = len(adata_pred.uns["cytokine_in_train"])
    expr = "_".join(pred_file.split("_")[:-1])
    df.to_csv(os.path.join(out_dir, f"{expr}_metrics_{k}.csv"))
    

INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              
INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              


/home/icb/dominik.klein/mambaforge/envs/cfp/lib/python3.11/site-packages/cfp/preprocessing/_wknn.py:90: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  ref_adata.uns[uns_key_added] = wknn


INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              
INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              
INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              
INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              
INFO     cuML is not installed or GPU is not available. Falling back to 

In [26]:
for k in deg_genes.keys():
    adata_pred.obs["cytokine"] = cytokine
    adata_pred.obs["donor"] = donor
    adata_pred.var_names=adata_ctrl.var_names 
    cond = f'{donor}_{cytokine}'
    out = compute_metrics(adata_ref=adata_ref, adata_pred=adata_pred, deg_dict=deg_genes, adata_ood_true=adata_ood_true, adata_ctrl=adata_ctrl)
    df = pd.DataFrame.from_dict(out, orient="index")
    df["cond"] = cond
    df["wandb_name"] = pred_file.split("_")[0]
    df["cytokine_in_train"] = len(adata_pred.uns["cytokine_in_train"])
    break

INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              
INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              


In [27]:
df = pd.DataFrame.from_dict(out)

In [30]:
df = pd.DataFrame.from_dict(out, orient="index")
df["cond"] = cond
df["wandb_name"] = pred_file.split("_")[0]
df["cytokine_in_train"] = len(adata_pred.uns["cytokine_in_train"])

In [31]:
df

,deg_decoded_r_squared_CD4 Naive,deg_decoded_r_squared_NKT,deg_decoded_r_squared_CD56-dim NK,deg_decoded_r_squared_CD4 Memory,deg_decoded_r_squared_MAIT,deg_decoded_r_squared_CD16 Mono,deg_decoded_r_squared_B Naive,deg_decoded_r_squared_B Intermediate/Memory,deg_decoded_r_squared_CD8 Memory,deg_decoded_r_squared_cDC,deg_decoded_r_squared_CD14 Mono,deg_decoded_r_squared_CD56-bright NK,deg_decoded_r_squared_CD8 Naive,deg_decoded_r_squared_Treg,cond,wandb_name,cytokine_in_train
20,0.837256,0.862121,0.914406,0.958514,0.904165,0.817752,0.715379,0.895184,0.845858,0.904087,0.954262,0.758896,0.727101,0.611289,Donor6_ADSF,absurd-puddle-164,65
50,0.924527,0.774764,0.914695,0.962029,0.885869,0.886041,0.836715,0.915801,0.828100,0.866120,0.913004,0.729895,0.801957,0.793504,Donor6_ADSF,absurd-puddle-164,65
100,0.936403,0.827012,0.942984,0.964499,0.909647,0.933248,0.903273,0.918539,0.870975,0.881387,0.927312,0.814896,0.803635,0.837237,Donor6_ADSF,absurd-puddle-164,65
200,0.957724,0.870379,0.946016,0.969604,0.921428,0.959747,0.943895,0.937650,0.902379,0.890245,0.956862,0.858010,0.866928,0.868276,Donor6_ADSF,absurd-puddle-164,65
500,0.966815,0.911572,0.960654,0.975808,0.940627,0.971049,0.964144,0.954678,0.930760,0.928938,0.973504,0.892722,0.896381,0.910034,Donor6_ADSF,absurd-puddle-164,65
1000,0.971952,0.928770,0.966943,0.979154,0.949443,0.974400,0.968151,0.961284,0.943668,0.943577,0.979616,0.907350,0.910901,0.923075,Donor6_ADSF,absurd-puddle-164,65


In [37]:
adata_pred = sc.read_h5ad(complete_pred_file)
adata_pred.X = adata_pred.layers["X_recon"]
cytokine = pred_file.split("_")[-3]
donor = pred_file.split("_")[-4]
out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/metrics_new_donor_different_k"
adata_full = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/pbmc_with_pca.h5ad")
adata_ood_true = adata_full[(adata_full.obs["donor"] == donor) & (adata_full.obs["cytokine"]==cytokine)]
adata_ctrl = adata_full[(adata_full.obs["cytokine"]=="PBS") & (adata_full.obs["donor"]==donor)]
with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/idcs_to_keep.pkl", "rb") as pickle_file:
    idcs_to_keep = pickle.load(pickle_file)

adata_ref = adata_full[adata_full.obs_names.isin(idcs_to_keep)]
project_pca(query_adata=adata_pred, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")


with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/degs_different_top_k.pkl", "rb") as pickle_file:
    deg_genes = pickle.load(pickle_file)

for k in deg_genes.keys():
    adata_pred.obs["cytokine"] = cytokine
    adata_pred.obs["donor"] = donor
    adata_pred.var_names=adata_ctrl.var_names   
    cond = f'{donor}_{cytokine}'
    out = compute_metrics(adata_ref=adata_ref, adata_pred=adata_pred, deg_dict=deg_genes, adata_ood_true=adata_ood_true, adata_ctrl=adata_ctrl)
    df = pd.DataFrame.from_dict(out)
    df["cond"] = cond
    df["wandb_name"] = pred_file.split("_")[0]
    df["cytokine_in_train"] = len(adata_pred.uns["cytokine_in_train"])
    expr = "_".join(pred_file.split("_")[:-1])
    df.to_csv(os.path.join(out_dir, f"{expr}_metrics_{k}.csv"))
    break

INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              
INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              


/home/icb/dominik.klein/mambaforge/envs/cfp/lib/python3.11/site-packages/cfp/preprocessing/_wknn.py:90: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  ref_adata.uns[uns_key_added] = wknn


In [38]:
df

,20,50,100,200,500,1000,cond,wandb_name,cytokine_in_train
deg_decoded_r_squared_CD4 Naive,0.833541,0.922571,0.934673,0.956614,0.965962,0.971273,Donor6_ADSF,absurd-puddle-164,65
deg_decoded_r_squared_NKT,0.861192,0.773698,0.825655,0.869893,0.911265,0.928176,Donor6_ADSF,absurd-puddle-164,65
deg_decoded_r_squared_CD56-dim NK,0.915164,0.914958,0.943114,0.946337,0.960972,0.967181,Donor6_ADSF,absurd-puddle-164,65
deg_decoded_r_squared_CD4 Memory,0.963007,0.964651,0.965241,0.970199,0.976199,0.979484,Donor6_ADSF,absurd-puddle-164,65
deg_decoded_r_squared_MAIT,0.897697,0.883254,0.907191,0.919357,0.939401,0.948561,Donor6_ADSF,absurd-puddle-164,65
deg_decoded_r_squared_CD16 Mono,0.818534,0.886069,0.933364,0.959830,0.971112,0.974480,Donor6_ADSF,absurd-puddle-164,65
deg_decoded_r_squared_B Naive,0.715516,0.836857,0.903113,0.943955,0.964050,0.968074,Donor6_ADSF,absurd-puddle-164,65
deg_decoded_r_squared_B Intermediate/Memory,0.895081,0.912920,0.917873,0.936367,0.954142,0.960787,Donor6_ADSF,absurd-puddle-164,65
deg_decoded_r_squared_CD8 Memory,0.834286,0.816900,0.863951,0.896929,0.927118,0.940982,Donor6_ADSF,absurd-puddle-164,65
deg_decoded_r_squared_cDC,0.901288,0.861844,0.879512,0.887113,0.927117,0.942505,Donor6_ADSF,absurd-puddle-164,65


In [39]:
df = pd.read_csv("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/metrics_new_donor_different_k/absurd-puddle-164_Donor6_IFN-gamma_65_metrics_50.csv")

In [40]:
df


,Unnamed: 0,20,50,100,200,500,1000,cond,wandb_name,cytokine_in_train
0,deg_decoded_r_squared_CD4 Naive,0.977027,0.975805,0.956721,0.961632,0.968109,0.970591,Donor6_IFN-gamma,absurd-puddle-164,65
1,deg_decoded_r_squared_NKT,0.314567,0.682230,0.752469,0.816545,0.869082,0.890984,Donor6_IFN-gamma,absurd-puddle-164,65
2,deg_decoded_r_squared_CD56-dim NK,0.848817,0.858996,0.910517,0.933666,0.954450,0.960132,Donor6_IFN-gamma,absurd-puddle-164,65
3,deg_decoded_r_squared_CD4 Memory,0.958364,0.962634,0.968354,0.970400,0.975038,0.976689,Donor6_IFN-gamma,absurd-puddle-164,65
4,deg_decoded_r_squared_MAIT,0.749822,0.814814,0.861803,0.905177,0.923375,0.930518,Donor6_IFN-gamma,absurd-puddle-164,65
5,deg_decoded_r_squared_CD16 Mono,0.553483,0.688777,0.797841,0.892593,0.936059,0.947488,Donor6_IFN-gamma,absurd-puddle-164,65
6,deg_decoded_r_squared_B Naive,0.855616,0.857612,0.881449,0.920553,0.948101,0.955275,Donor6_IFN-gamma,absurd-puddle-164,65
7,deg_decoded_r_squared_B Intermediate/Memory,0.872647,0.879705,0.899223,0.911358,0.918828,0.940704,Donor6_IFN-gamma,absurd-puddle-164,65
8,deg_decoded_r_squared_CD8 Memory,0.695250,0.805640,0.883851,0.917103,0.934804,0.944922,Donor6_IFN-gamma,absurd-puddle-164,65
9,deg_decoded_r_squared_cDC,-0.257601,0.331657,0.587781,0.761144,0.867097,0.894967,Donor6_IFN-gamma,absurd-puddle-164,65


In [20]:
data_dir = "/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/metrics_new_donor"

In [21]:
import pandas as pd
import os
import h5py
import seaborn as sns
import scanpy as sc
import anndata as ad
from datetime import datetime

/home/icb/dominik.klein/mambaforge/envs/cfp/lib/python3.11/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module


In [22]:
def get_cytokine_in_train(file: str) -> int:
    PREDS_DIR = "/lustre/groups/ml01/workspace/ot_perturbation/models/otfm/pbmc_new_donor"
    file_path = os.path.join(PREDS_DIR, f"{file}_preds.h5ad")
    with h5py.File(file_path, "r") as f:
    
        if "uns" in f:
            uns = {}
            for key in f["uns"].keys():
                uns[key] = f["uns"][key][()]
        else:
            raise ValueError(f"Error for {file}")

    out = uns['cytokine_in_train']
    return [out[i].decode('UTF-8') for i in range(len(out))]

In [23]:
dfs = []
for el in os.listdir(data_dir):
    if el.endswith("metrics.csv"):
        file_path = os.path.join(data_dir, el)
        df_tmp = pd.read_csv(file_path, index_col=0).T
        df_tmp["donor_cytokine"] = df_tmp.index
        df_tmp["donor"] = df_tmp.apply(lambda x: x["donor_cytokine"].split("_")[0], axis=1)
        df_tmp["cytokine"] = df_tmp.apply(lambda x: x["donor_cytokine"].split("_")[1], axis=1)
        file = f"{df_tmp['wandb_name'].values[0]}_{df_tmp['donor'].values[0]}_{df_tmp['cytokine'].values[0]}_{df_tmp['cytokine_in_train'].values[0]}"
        df_tmp["num_cytokines_in_train"] = df_tmp["cytokine_in_train"]
        df_tmp["cytokine_in_train"] = str(get_cytokine_in_train(file))
        modification_time = os.path.getmtime(file_path)
        df_tmp["date"] = datetime.fromtimestamp(modification_time).strftime('%Y-%m-%d %H:%M:%S')
        
        dfs.append(df_tmp)
        


In [24]:
len(dfs)

2880

In [25]:
df = pd.concat(dfs)

In [26]:
for el in df.columns:
    try:
        df[el] = df[el].astype("float")
    except:
        continue

In [27]:
df["donor_cytokine"] = df.index
df["donor"] = df.apply(lambda x: x["donor_cytokine"].split("_")[0], axis=1)
df["cytokine"] = df.apply(lambda x: x["donor_cytokine"].split("_")[1], axis=1)

In [28]:
df["num_cytokines_in_train"]

Donor5_M-CSF          33.0
Donor7_FasL           65.0
Donor12_IL-32-beta     2.0
Donor3_FasL           33.0
Donor2_OX40L          17.0
                      ... 
Donor3_CD27L           2.0
Donor2_M-CSF           9.0
Donor5_OX40L          17.0
Donor8_IFN-omega       9.0
Donor2_BAFF            1.0
Name: num_cytokines_in_train, Length: 2880, dtype: float64

In [29]:
df_other = df[df["num_cytokines_in_train"]!=1.0]
df_1 = df[df["num_cytokines_in_train"]==1.0]
df_1["new_run"] = df_1.apply(lambda x: True if x["date"]>"2025-01-20" else False, axis=1)
df_1["new_run"].value_counts()

/tmp/ipykernel_1470393/386908205.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_1["new_run"] = df_1.apply(lambda x: True if x["date"]>"2025-01-20" else False, axis=1)
/tmp/ipykernel_1470393/386908205.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_1["new_run"] = df_1.apply(lambda x: True if x["date"]>"2025-01-20" else False, axis=1)


new_run
True     120
False    120
Name: count, dtype: int64